In [2]:
import os
import random
from pathlib import Path
from urllib.request import urlretrieve
import requests
from pathlib import Path
from tqdm.auto import tqdm
import dagshub
import mlflow
import mlflow.pytorch
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision.transforms as T
from PIL import Image
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from torchvision.models import EfficientNet_B0_Weights
from tqdm.auto import tqdm

Тут будем пробовать новую архитектуру EfficientNet. У нее есть несколько версий, различающихся по величине и сложности от B0 до B3 и дальше. Для начала подготовим данные

In [3]:
def seed_everything(seed):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)


seed_everything(42)

In [ ]:
class CFG:
    seed = 42
    image_col = "image_link"
    target_col = "price"
    cache_dir = "../data/images_cache"
    img_size = 224
    batch_size = 32
    num_workers = 0
    epochs = 5
    lr = 1e-4
    weight_decay = 1e-5
    val_size = 0.2
    artifacts_dir = "model_outputs/efficientnet_b0",
    model_type = 'EfficientNet-B0'
    model_name = "efficientnet_b0.pt"
    experiment_name = "amazon_smart_pricing"
    run_name = "efficientnet_b0_images"

In [ ]:
dagshub.init(
    repo_owner="Dezurn",
    repo_name="Deep-Learning",
    mlflow=True,
)

mlflow.set_experiment(CFG.experiment_name)

In [6]:
train_df = pd.read_csv("../data/raw/train.csv")
test_df = pd.read_csv("../data/raw/test.csv")
train_df.head()

,sample_id,catalog_content,image_link,price
0,33127,"Item Name: La Victoria Green Taco Sauce Mild, ...",https://m.media-amazon.com/images/I/51mo8htwTH...,4.89
1,198967,"Item Name: Salerno Cookies, The Original Butte...",https://m.media-amazon.com/images/I/71YtriIHAA...,13.12
2,261251,"Item Name: Bear Creek Hearty Soup Bowl, Creamy...",https://m.media-amazon.com/images/I/51+PFEe-w-...,1.97
3,55858,Item Name: Judee’s Blue Cheese Powder 11.25 oz...,https://m.media-amazon.com/images/I/41mu0HAToD...,30.34
4,292686,"Item Name: kedem Sherry Cooking Wine, 12.7 Oun...",https://m.media-amazon.com/images/I/41sA037+Qv...,66.49


Загружать такое количество картинок очень долго и для экспериментов можно взять куда меньшее количество данных.

In [7]:
num = 5000
work_df = train_df.sample(num, random_state=CFG.seed).reset_index(drop=True)
val_data = work_df.iloc[: int(len(work_df) * CFG.val_size)].reset_index(drop=True)
train_data = work_df.iloc[int(len(work_df) * CFG.val_size) :].reset_index(drop=True)

In [8]:
len(val_data), len(train_data)

(1000, 4000)

In [9]:
from urllib.request import urlretrieve
from pathlib import Path
from tqdm.auto import tqdm
def download_images_simple(df):
    Path(CFG.cache_dir).mkdir(parents=True, exist_ok=True)

    image_paths = []

    for url in tqdm(df[CFG.image_col], desc="Downloading images"):
        filename = str(url).split("/")[-1].split("?")[0]
        image_path = f"{CFG.cache_dir}/{filename}"

        try:
            if not Path(image_path).exists():
                urlretrieve(url, image_path)

            image_paths.append(image_path)

        except Exception:
            image_paths.append(None)

    df = df.copy()
    df["image_path"] = image_paths
    df = df.dropna(subset=["image_path"]).reset_index(drop=True)

    return df

In [10]:
import requests
from pathlib import Path
from tqdm.auto import tqdm

def download_images_simple(df):
    Path(CFG.cache_dir).mkdir(parents=True, exist_ok=True)

    image_paths = []
    errors = []

    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/126.0.0.0 Safari/537.36"
        )
    }

    for url in tqdm(df[CFG.image_col], desc="Downloading images"):
        filename = str(url).split("/")[-1].split("?")[0]
        image_path = Path(CFG.cache_dir) / filename

        try:
            if not image_path.exists():
                response = requests.get(url, headers=headers, timeout=20)
                response.raise_for_status()

                with open(image_path, "wb") as f:
                    f.write(response.content)

            image_paths.append(str(image_path))

        except Exception as e:
            image_paths.append(None)
            errors.append((url, str(e)))

    df = df.copy()
    df["image_path"] = image_paths

    print("Всего строк:", len(df))
    print("Скачано/найдено картинок:", df["image_path"].notna().sum())
    print("Ошибок:", len(errors))

    if errors:
        print("Пример ошибки:")
        print(errors[0])

    df = df.dropna(subset=["image_path"]).reset_index(drop=True)

    return df

In [11]:
train_data = download_images_simple(train_data)
val_data = download_images_simple(val_data)

Всего строк: 4000
Скачано/найдено картинок: 4000
Ошибок: 0


Всего строк: 1000
Скачано/найдено картинок: 1000
Ошибок: 0


In [12]:
train_data.head(), val_data.head()

(   sample_id                                    catalog_content  \
 0     156787  Item Name: 'Rich JW Allen Snow Queen Icing Bas...   
 1     225070  Item Name: Dandies Vegan Marshmallows, Vanilla...   
 2      46913  Item Name: Canels Gum Box Original 60 Count\r\...   
 3     298119  Item Name: HERSHEY'S Miniatures Assorted Choco...   
 4      68001  Item Name: Pink Lemonade Licorice Sour Sticks ...   
 
                                           image_link    price  \
 0  https://m.media-amazon.com/images/I/71mDwF1ANo...  113.310   
 1  https://m.media-amazon.com/images/I/71hAUfz93S...    5.490   
 2  https://m.media-amazon.com/images/I/91DIhiMcqI...    4.035   
 3  https://m.media-amazon.com/images/I/51iYRznRks...   20.790   
 4  https://m.media-amazon.com/images/I/91hRyLkNi+...   18.990   
 
                              image_path  
 0  ..\data\images_cache\71mDwF1ANoL.jpg  
 1  ..\data\images_cache\71hAUfz93SL.jpg  
 2  ..\data\images_cache\91DIhiMcqIL.jpg  
 3  ..\data\images_c

In [13]:
class ImagePriceDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image = Image.open(row["image_path"]).convert("RGB")

        if self.transform:
            image = self.transform(image)

        price = np.log1p(row[CFG.target_col]).astype("float32")

        return image, torch.tensor(price, dtype=torch.float32)

Можно взять предобученные параметры из IMAGENET для картинок для более быстрого обучения.

In [ ]:

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
train_transform = T.Compose([
    T.Resize(256), # меняем размер
    T.RandomCrop(CFG.img_size), # вырезаем случайную область указанного размера
    T.RandomHorizontalFlip(), # зеркальное отражение по горизонтали
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD) # нормализация
])

valid_transform = T.Compose([ # на валидации нельзя делать никаких преобразований (кроме изменения размера),
                               # ведь мы должны проверить модель на реальных картинках
    T.Resize(256),
    T.CenterCrop(CFG.img_size),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

In [15]:
train_dataset = ImagePriceDataset(train_data, transform=train_transform)
val_dataset = ImagePriceDataset(val_data, transform=valid_transform)

train_loader = DataLoader(train_dataset, batch_size=CFG.batch_size, shuffle=True, num_workers=CFG.num_workers)

val_loader = DataLoader(val_dataset, batch_size=CFG.batch_size, shuffle=False, num_workers=CFG.num_workers)

In [16]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

Для начала создадим EfficientNetB0, это самая быстрая и легкая модель. Будем брать предобученные веса, так как это помогает модели быстрее учиться 

In [ ]:
class EfficientNetB0PriceRegressor(nn.Module):
    def __init__(self):
        super().__init__()

        weights = EfficientNet_B0_Weights.DEFAULT
        self.model = models.efficientnet_b0(weights=weights)

        for param in self.model.features.parameters():
            param.requires_grad = False

        in_features = self.model.classifier[1].in_features
        self.model.classifier[1] = nn.Linear(in_features, 1)

    def forward(self, x):
        return self.model(x).squeeze(1)


model_efficientnet_b0 = EfficientNetB0PriceRegressor().to(device)
model_efficientnet_b0

EfficientNetB0PriceRegressor(
  (model): EfficientNet(
    (features): Sequential(
      (0): Conv2dNormActivation(
        (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (2): SiLU(inplace=True)
      )
      (1): Sequential(
        (0): MBConv(
          (block): Sequential(
            (0): Conv2dNormActivation(
              (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
              (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
              (2): SiLU(inplace=True)
            )
            (1): SqueezeExcitation(
              (avgpool): AdaptiveAvgPool2d(output_size=1)
              (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
              (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
              (activation): SiLU

In [ ]:
criterion = nn.MSELoss()

optimizer = torch.optim.AdamW(
    filter(lambda param: param.requires_grad, model_efficientnet_b0.parameters()),
    lr=CFG.lr,
    weight_decay=CFG.weight_decay,
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2,
)

In [18]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()

    running_loss = 0.0

    for images, targets in tqdm(loader, desc="Train", leave=False):
        images = images.to(device)
        targets = targets.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, targets)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

    epoch_loss = running_loss / len(loader.dataset)

    return epoch_loss

In [19]:
def validate_one_epoch(model, loader, criterion, device):
    model.eval()

    running_loss = 0.0

    preds = []
    targets_all = []

    with torch.no_grad():
        for images, targets in tqdm(loader, desc="Valid", leave=False):
            images = images.to(device)
            targets = targets.to(device)

            outputs = model(images)
            loss = criterion(outputs, targets)

            running_loss += loss.item() * images.size(0)

            preds.extend(outputs.detach().cpu().numpy())
            targets_all.extend(targets.detach().cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)

    preds = np.array(preds)
    targets_all = np.array(targets_all)

    preds_price = np.expm1(preds)
    targets_price = np.expm1(targets_all)

    preds_price = np.clip(preds_price, 0, None)

    mae = mean_absolute_error(targets_price, preds_price)
    mse = mean_squared_error(targets_price, preds_price)
    rmse = np.sqrt(mse)
    r2 = r2_score(targets_price, preds_price)

    metrics = {
        "val_loss": epoch_loss,
        "mae": mae,
        "rmse": rmse,
        "r2": r2,
    }

    return metrics

In [30]:
def fit_model(model, train_loader, val_loader, optimizer, scheduler, criterion, device):
    best_rmse = float("inf")
    history = []

    os.makedirs(CFG.artifacts_dir, exist_ok=True)
    best_model_path = os.path.join(CFG.artifacts_dir, CFG.model_name)
    history_path = os.path.join(CFG.artifacts_dir, "efficientnet_history.csv")

    mlflow.log_params(
        {
            "model": CFG.model_type,
            "pretrained": True,
            "frozen_backbone": True,
            "target": "log1p(price)",
            "img_size": CFG.img_size,
            "batch_size": CFG.batch_size,
            "epochs": CFG.epochs,
            "lr": CFG.lr,
            "weight_decay": CFG.weight_decay,
            "optimizer": "AdamW",
            "loss": "MSELoss",
            "scheduler": "ReduceLROnPlateau",
            "val_size": CFG.val_size,
            "seed": CFG.seed,
        }
    )

    for epoch in range(1, CFG.epochs + 1):
        print(f"\nEpoch {epoch}/{CFG.epochs}")

        train_loss = train_one_epoch(
            model,
            train_loader,
            optimizer,
            criterion,
            device,
        )

        val_metrics = validate_one_epoch(
            model,
            val_loader,
            criterion,
            device,
        )

        scheduler.step(val_metrics["val_loss"])

        row = {
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_metrics["val_loss"],
            "mae": val_metrics["mae"],
            "rmse": val_metrics["rmse"],
            "r2": val_metrics["r2"],
        }

        history.append(row)

        mlflow.log_metrics(row, step=epoch)

        print(
            f"train_loss: {train_loss:.4f} | "
            f"val_loss: {val_metrics['val_loss']:.4f} | "
            f"MAE: {val_metrics['mae']:.2f} | "
            f"RMSE: {val_metrics['rmse']:.2f} | "
            f"R2: {val_metrics['r2']:.4f}"
        )

        if val_metrics["rmse"] < best_rmse:
            best_rmse = val_metrics["rmse"]

            torch.save(
                {
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "epoch": epoch,
                    "best_rmse": best_rmse,
                },
                best_model_path,
            )

            print(f"Best model saved: {best_model_path}")

    history = pd.DataFrame(history)
    history.to_csv(history_path, index=False)
    
    best_checkpoint = torch.load(best_model_path, map_location=device)
    model.load_state_dict(best_checkpoint["model_state_dict"])

    mlflow.log_metric("best_rmse", best_rmse)
    mlflow.log_artifact(best_model_path, artifact_path="model_checkpoint")
    mlflow.log_artifact(history_path, artifact_path="history")
    mlflow.pytorch.log_model(model, artifact_path="model")

    return model, history

In [111]:
with mlflow.start_run(run_name=CFG.run_name):
    model_efficientnet_b0, history_efficientnet_b0 = fit_model(
        model=model_efficientnet_b0,
        train_loader=train_loader,
        val_loader=val_loader,
        optimizer=optimizer,
        scheduler=scheduler,
        criterion=criterion,
        device=device,
    )


Epoch 1/5


Train:   0%|          | 0/125 [00:00<?, ?it/s]

Valid:   0%|          | 0/32 [00:00<?, ?it/s]

train_loss: 4.7750 | val_loss: 3.0793 | MAE: 19.84 | RMSE: 35.60 | R2: -0.3923
Best model saved: model_outputs/efficientnet_b0\efficientnet_b0.pt

Epoch 2/5


Train:   0%|          | 0/125 [00:00<?, ?it/s]

...

Train:   0%|          | 0/125 [00:00<?, ?it/s]

Valid:   0%|          | 0/32 [00:00<?, ?it/s]

train_loss: 2.0629 | val_loss: 2.2442 | MAE: 19.68 | RMSE: 35.08 | R2: -0.3514
Best model saved: model_outputs/efficientnet_b0\efficientnet_b0.pt


2026/06/14 21:00:04 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/14 21:00:11 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.


🏃 View run efficientnet_b0_images at: https://dagshub.com/Dezurn/Deep-Learning.mlflow/#/experiments/1/runs/06cd9a7e48a64de297fbde11b14d9404
🧪 View experiment at: https://dagshub.com/Dezurn/Deep-Learning.mlflow/#/experiments/1


Получили неплохое качество. теперь давайте возьмем модель потяжелее. Перейдем на B2

In [28]:
class CFG:
    seed = 42
    image_col = "image_link"
    target_col = "price"
    cache_dir = "../data/images_cache"
    img_size = 224
    batch_size = 32
    num_workers = 0
    epochs = 5
    lr = 1e-4
    weight_decay = 1e-5
    val_size = 0.2
    artifacts_dir = "model_outputs/efficientnet_b2"
    model_name = "efficientnet_b2.pt"
    model_type = 'EfficientNet-B2'
    experiment_name = "amazon_smart_pricing"
    run_name = "efficientnet_b2_images"

In [22]:
dagshub.init(
    repo_owner="Dezurn",
    repo_name="Deep-Learning",
    mlflow=True,
)

mlflow.set_experiment(CFG.experiment_name)

Initialized MLflow to track repo "Dezurn/Deep-Learning"

Repository Dezurn/Deep-Learning initialized!

<Experiment: artifact_location='mlflow-artifacts:/99b0aadd413f4a00bf17d371a1a7a7aa', creation_time=1781452541346, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1781452541346, lifecycle_stage='active', name='amazon_smart_pricing', tags={'mlflow.experimentKind': 'custom_model_development'}, trace_location=None, workspace='default'>

In [25]:
from torchvision.models import EfficientNet_B2_Weights

class EfficientNetB2PriceRegressor(nn.Module):
    def __init__(self):
        super().__init__()

        weights = EfficientNet_B2_Weights.DEFAULT
        self.model = models.efficientnet_b2(weights=weights)

        for param in self.model.features.parameters():
            param.requires_grad = False

        in_features = self.model.classifier[1].in_features
        self.model.classifier[1] = nn.Linear(in_features, 1)

    def forward(self, x):
        return self.model(x).squeeze(1)


model_efficientnet_b2 = EfficientNetB2PriceRegressor().to(device)
model_efficientnet_b2

Downloading: "https://download.pytorch.org/models/efficientnet_b2_rwightman-c35c1473.pth" to C:\Users\miste/.cache\torch\hub\checkpoints\efficientnet_b2_rwightman-c35c1473.pth
100%|██████████| 35.2M/35.2M [00:01<00:00, 19.5MB/s]


EfficientNetB2PriceRegressor(
  (model): EfficientNet(
    (features): Sequential(
      (0): Conv2dNormActivation(
        (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): SiLU(inplace=True)
      )
      (1): Sequential(
        (0): MBConv(
          (block): Sequential(
            (0): Conv2dNormActivation(
              (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
              (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
              (2): SiLU(inplace=True)
            )
            (1): SqueezeExcitation(
              (avgpool): AdaptiveAvgPool2d(output_size=1)
              (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
              (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
              (activation): SiLU(inplace=True)
       

Здесь менять ничего не будем

In [26]:
optimizer_b2 = torch.optim.AdamW(
    filter(lambda param: param.requires_grad, model_efficientnet_b2.parameters()),
    lr=CFG.lr,
    weight_decay=CFG.weight_decay,
)

scheduler_b2 = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_b2,
    mode="min",
    factor=0.5,
    patience=2,
)

In [29]:
with mlflow.start_run(run_name=CFG.run_name):
    model_efficientnet_b2, history_efficientnet_b2 = fit_model(
        model=model_efficientnet_b2,
        train_loader=train_loader,
        val_loader=val_loader,
        optimizer=optimizer_b2,
        scheduler=scheduler_b2,
        criterion=criterion,
        device=device,
    )


Epoch 1/5


Train:   0%|          | 0/125 [00:00<?, ?it/s]

Valid:   0%|          | 0/32 [00:00<?, ?it/s]

train_loss: 5.0050 | val_loss: 3.5213 | MAE: 20.43 | RMSE: 36.09 | R2: -0.4312
Best model saved: model_outputs/efficientnet_b2\efficientnet_b2.pt

Epoch 2/5


Train:   0%|          | 0/125 [00:00<?, ?it/s]

...

Train:   0%|          | 0/125 [00:00<?, ?it/s]

Valid:   0%|          | 0/32 [00:00<?, ?it/s]

train_loss: 2.0319 | val_loss: 2.4627 | MAE: 18.90 | RMSE: 34.42 | R2: -0.3020
Best model saved: model_outputs/efficientnet_b2\efficientnet_b2.pt


2026/06/14 22:37:18 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/14 22:37:24 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
2026/06/14 22:37:24 WARNING mlflow.utils.requirements_utils: Found torch version (2.5.1+cu121) contains a local version label (+cu121). MLflow logged a pip requirement for this package as 'torch==2.5.1' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/06/14 22:37:29 WARNING mlflow.utils.requirements_utils: Found torchvision version (0.20.1+cu121) contains a local version l

🏃 View run efficientnet_b2_images at: https://dagshub.com/Dezurn/Deep-Learning.mlflow/#/experiments/1/runs/6c8a031f4ee243ddb45daa286c3fefec
🧪 View experiment at: https://dagshub.com/Dezurn/Deep-Learning.mlflow/#/experiments/1


Давайте попробуем большее количество эпох, так как еще нет признаков переобучения

In [31]:
CFG.epochs = 10

In [32]:
with mlflow.start_run(run_name=CFG.run_name):
    model_efficientnet_b2, history_efficientnet_b2 = fit_model(
        model=model_efficientnet_b2,
        train_loader=train_loader,
        val_loader=val_loader,
        optimizer=optimizer_b2,
        scheduler=scheduler_b2,
        criterion=criterion,
        device=device,
    )


Epoch 1/10


Train:   0%|          | 0/125 [00:00<?, ?it/s]

Valid:   0%|          | 0/32 [00:00<?, ?it/s]

train_loss: 1.9375 | val_loss: 2.3995 | MAE: 18.85 | RMSE: 34.37 | R2: -0.2985
Best model saved: model_outputs/efficientnet_b2\efficientnet_b2.pt

Epoch 2/10


Train:   0%|          | 0/125 [00:00<?, ?it/s]

...

Train:   0%|          | 0/125 [00:00<?, ?it/s]

Valid:   0%|          | 0/32 [00:00<?, ?it/s]

train_loss: 1.6084 | val_loss: 1.9266 | MAE: 18.04 | RMSE: 33.73 | R2: -0.2506
Best model saved: model_outputs/efficientnet_b2\efficientnet_b2.pt


C:\Users\miste\AppData\Local\Temp\ipykernel_23844\2869801139.py:87: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  best_checkpoint = torch.load(best_model_path, map_location=

🏃 View run efficientnet_b2_images at: https://dagshub.com/Dezurn/Deep-Learning.mlflow/#/experiments/1/runs/ecaa761a62604c2b895214d9d8e6f24b
🧪 View experiment at: https://dagshub.com/Dezurn/Deep-Learning.mlflow/#/experiments/1
